# Assignment 4: Energy-Based Models and Score-Based Models
In this assignment, you will implement energy-based models (EBMs) and score-based generative models, and run them on two distinct datasets:
- **Dataset 1 (Pinwheel)**: A highly non-linear 2D spiral whirlpool dataset.
- **Dataset 2 (GMM 3D)**: A 3D Gaussian Mixture Model with 4 components.

All generated plots will be saved in the `results/` folder.

**After you complete the assignment, submit the images outputted in the `results/` folder along with the final test/val losses.**

First, execute the cell below to set up devices and load the libraries.


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

from pytorch_util import device, from_numpy, get_numpy
from hw4_helper import (
    load_data,
    visualize_data,
    q1_save_results,
    q2_a_save_results,
    q2_b_save_results
)


In [2]:
# Load both Pinwheel and 3D GMM datasets
train_data_1, val_data_1 = load_data(dset_type=1)
train_data_2, val_data_2 = load_data(dset_type=2)

# Visualize Dataset 1 (2D Pinwheel)
visualize_data(dset_type=1, data=train_data_1)

# Visualize Dataset 2 (3D GMM)
visualize_data(dset_type=2, data=train_data_2)

# Question 1: Energy-Based Models (EBMs)
In this question, you will implement and train a simple Energy-Based Model using Contrastive Divergence (CD-k).

The model's probability density is defined as:
$$p_\theta(x) = \frac{\exp(-E_\theta(x))}{Z_\theta}$$

where $E_\theta(x)$ is a neural network mapping $x \in \mathbb{R}^d \to \mathbb{R}$ ($d$ is `in_dim`), and $Z_\theta$ is the normalizing constant.

Since $Z_\theta$ is intractable, you will use Contrastive Divergence (CD-k) to approximate the gradient of the $negative$ log-likelihood:
$$\nabla_\theta \mathcal{L}_{CD} \approx \frac{1}{B} \sum_{i=1}^B \nabla_\theta E_\theta(x_{pos}^{(i)}) - \frac{1}{B} \sum_{i=1}^B \nabla_\theta E_\theta(x_{neg}^{(i)})$$

To generate the negative samples $x_{neg}$, you will run $k$ steps of MCMC starting from positive data samples $x_{pos}$. In order to match the training and generation phases, you will support both:
1. **Metropolis-Hastings (MH) MCMC**: Propose candidates $x_{prop} = x + \text{std} \cdot \mathcal{N}(0, I)$ and accept them with probability $\min(1, \exp(-E_\theta(x_{prop}) + E_\theta(x_{curr})))$.
2. **Unadjusted Langevin MCMC**: Update using gradients of the energy network:
$$x_{t+1} = x_t - \frac{\alpha}{2} \nabla_x E_\theta(x_t) + \sqrt{\alpha} z_t, \quad z_t \sim \mathcal{N}(0, I)$$

### Tasks:
1. Implement Metropolis-Hastings MCMC sampling (`sample_mh`).
2. Implement Langevin MCMC sampling (`sample_langevin`).
3. Implement the Contrastive Divergence loss computation and MCMC negative phase (both MH and Langevin based on `use_langevin`) inside `train_ebm`.


In [3]:
class EnergyNet(nn.Module):
    def __init__(self, in_dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.SiLU(),
            nn.Linear(128, 128),
            nn.SiLU(),
            nn.Linear(128, 128),
            nn.SiLU(),
            nn.Linear(128, 1)
        )
        
    def forward(self, x):
        return self.net(x)


In [ ]:
def sample_mh(model, num_samples, in_dim=2, num_steps=500, prop_std=0.2):
    model.eval()
    # TODO: Implement Metropolis-Hastings MCMC sampling from EBM model supporting dynamic in_dim
    # 1. Initialize samples uniformly from [-4, 4] in R^in_dim.
    # 2. For each step, propose candidate: x_prop = x + prop_std * N(0, I)
    # 3. Calculate acceptance ratio: exp(-E(x_prop) + E(x_current))
    # 4. Accept/reject candidates and update.
    x = torch.FloatTensor(num_samples, in_dim).uniform_(-4, 4).to(device)

    with torch.no_grad():
        for _ in range(num_steps):
            x_prop = x + prop_std * torch.randn_like(x)
            e_curr = model(x).squeeze(-1)
            e_prop = model(x_prop).squeeze(-1)
            log_accept = -e_prop + e_curr
            accept = torch.rand(num_samples, device=device) < torch.exp(log_accept.clamp(max=0))
            x = torch.where(accept.unsqueeze(-1), x_prop, x)

    return get_numpy(x)

def sample_langevin(model, num_samples, in_dim=2, num_steps=500, alpha=0.01):
    model.eval()
    # TODO: Implement Langevin MCMC sampling from EBM model supporting dynamic in_dim
    # 1. Initialize samples uniformly from [-4, 4] in R^in_dim.
    # 2. For each step, compute grad_x of E(x) with respect to x.
    # 3. Update: x = x - 0.5 * alpha * grad_x + sqrt(alpha) * N(0, I)
    # Remember to set requires_grad=True and detach the chain at each step.
    x = torch.FloatTensor(num_samples, in_dim).uniform_(-4, 4).to(device)

    for _ in range(num_steps):
        x = x.detach().requires_grad_(True)
        energy = model(x).sum()
        grad = torch.autograd.grad(energy, x)[0]
        with torch.no_grad():
            x = x - 0.5 * alpha * grad + (alpha ** 0.5) * torch.randn_like(x)

    return get_numpy(x.detach())

def train_ebm(train_data, val_data, dset_type, use_langevin=True):
    import torch.optim as optim
    from torch.utils.data import DataLoader, TensorDataset
    
    in_dim = train_data.shape[1]
    model = EnergyNet(in_dim=in_dim).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    
    train_dataset = TensorDataset(torch.from_numpy(train_data))
    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
    
    train_losses = []
    val_losses = []
    
    # Dynamically select parameters based on dataset type
    if dset_type == 1: # 2D Pinwheel (Whirlpool)
        epochs = 30
        alpha = 0.02
        prop_std = 0.2
        k = 15
    else: # 3D GMM
        epochs = 20
        alpha = 0.05
        prop_std = 0.2
        k = 10
        
    for epoch in range(epochs):
        model.train()
        epoch_losses = []
        for (x_batch,) in train_loader:
            x_batch = x_batch.to(device)
            
            x_neg = x_batch.clone().detach()
            if use_langevin:
                # TODO: Generate negative samples x_neg using k steps of Langevin dynamics starting from x_batch
                # Temporarily enable gradients to compute ∇_x E(x) for the Langevin step, then detach to avoid backpropagating through the MCMC trajectory.
                for _ in range(k):
                    x_neg = x_neg.detach().requires_grad_(True)
                    energy = model(x_neg).sum()
                    grad = torch.autograd.grad(energy, x_neg)[0]
                    with torch.no_grad():
                        x_neg = x_neg - 0.5 * alpha * grad + (alpha ** 0.5) * torch.randn_like(x_neg)
                x_neg = x_neg.detach()
            else:
                # TODO: Generate negative samples x_neg using k steps of Metropolis-Hastings starting from x_batch
                # Hint: Propose candidate x_prop = x_neg + prop_std * N(0, I). Accept/reject candidates using exp(-E(x_prop) + E(x_neg)). And detach the gradient. 
                with torch.no_grad():
                    for _ in range(k):
                        x_prop = x_neg + prop_std * torch.randn_like(x_neg)
                        e_curr = model(x_neg).squeeze(-1)
                        e_prop = model(x_prop).squeeze(-1)
                        log_accept = -e_prop + e_curr
                        accept = torch.rand(x_neg.shape[0], device=device) < torch.exp(log_accept.clamp(max=0))
                        x_neg = torch.where(accept.unsqueeze(-1), x_prop, x_neg)
                x_neg = x_neg.detach()
            
            # TODO: Compute E(x_pos) and E(x_neg)
            energy_pos = model(x_batch)
            energy_neg = model(x_neg)
            
            # TODO: Since EBM energy values can grow unbounded, an L2 regularization term is pre-provided below to stabilize training.
            # Fill in the Contrastive Divergence loss term: (E(x_pos) - E(x_neg)).mean()
            cd_loss = (energy_pos - energy_neg).mean()
            loss = cd_loss + 0.1 * ((energy_pos ** 2).mean() + (energy_neg ** 2).mean())

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_losses.append(loss.item())
            
        model.eval()
        with torch.no_grad():
            x_val = torch.from_numpy(val_data).to(device)
            val_energy = model(x_val).mean()
            val_losses.append(val_energy.item())
            
        train_losses.append(np.mean(epoch_losses))
        
        if (epoch + 1) % 5 == 0:
            name = "Langevin" if use_langevin else "MH"
            print(f"Epoch {epoch+1:02d} ({name}) | Train CD Loss: {train_losses[-1]:.4f} | Val Energy: {val_losses[-1]:.4f}")
            
    # Sample after training using matched sampler
    if use_langevin:
        samples = sample_langevin(model, 1000, in_dim=in_dim, num_steps=500, alpha=0.01)
    else:
        samples = sample_mh(model, 1000, in_dim=in_dim, num_steps=500, prop_std=0.2)
    
    return model, train_losses, val_losses, samples

In [5]:
# Train EBM with Metropolis-Hastings MCMC on Dataset 1 (2D Pinwheel)
ebm_model_1_mh = q1_save_results(1, train_data_1, val_data_1, train_ebm, use_langevin=False)

# Train EBM with Langevin MCMC on Dataset 1 (2D Pinwheel)
ebm_model_1_langevin = q1_save_results(1, train_data_1, val_data_1, train_ebm, use_langevin=True)

# Train EBM with Metropolis-Hastings MCMC on Dataset 2 (3D GMM)
ebm_model_2_mh = q1_save_results(2, train_data_2, val_data_2, train_ebm, use_langevin=False)

# Train EBM with Langevin MCMC on Dataset 2 (3D GMM)
ebm_model_2_langevin = q1_save_results(2, train_data_2, val_data_2, train_ebm, use_langevin=True)


Training EBM with MH CD on Dataset 1 (Pinwheel (2D))...


Epoch 05 (MH) | Train CD Loss: -0.0523 | Val Energy: 0.0375


Epoch 10 (MH) | Train CD Loss: -0.2049 | Val Energy: 0.0733


Epoch 15 (MH) | Train CD Loss: -0.2920 | Val Energy: -0.1236


Epoch 20 (MH) | Train CD Loss: -0.3021 | Val Energy: -0.4739


...

Epoch 05 (Langevin) | Train CD Loss: -0.2096 | Val Energy: -0.4477


Epoch 10 (Langevin) | Train CD Loss: -0.2217 | Val Energy: -0.3662


Epoch 15 (Langevin) | Train CD Loss: -0.2166 | Val Energy: -0.0911


Epoch 20 (Langevin) | Train CD Loss: -0.2361 | Val Energy: -0.1545
Final Train Loss: -0.2361
Final Val Loss: -0.1545


Q1 (Langevin, Dataset 2) results saved successfully!


# Question 2: Score-Based Generative Models
Score-based generative models model the score function of the data distribution directly:
$$s_\theta(x) \approx \nabla_x \log p(x)$$

You will represent the score function as a neural network $s_\theta(x): \mathbb{R}^d \to \mathbb{R}^d$ ($d$ is `in_dim`).

### Question 2(a): Explicit Score Matching (Fisher Divergence)
In Explicit Score Matching, you minimize the Fisher divergence between the model's score function and the true data score:
$$\mathcal{L}_{ESM}(\theta) = \mathbb{E}_{p_{data}(x)} \left[ \frac{1}{2} \| s_\theta(x) \|_2^2 + \text{tr}(\nabla_x s_\theta(x)) \right]$$

Since $x \in \mathbb{R}^d$, the Jacobian trace is computed by summing the diagonal components:
$$\text{tr}(\nabla_x s_\theta(x)) = \sum_{d'=1}^d \frac{\partial s_{\theta, d'}(x)}{\partial x_{d'}}$$

### Tasks:
1. Implement Explicit Score Matching loss (`esm_loss`). Compute the Jacobian trace vectorially over the batch, supporting dynamic `in_dim`.
2. Implement Langevin MCMC sampling for score networks (`sample_score_langevin`).


In [6]:
class ScoreNet(nn.Module):
    def __init__(self, in_dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.SiLU(),
            nn.Linear(128, 128),
            nn.SiLU(),
            nn.Linear(128, 128),
            nn.SiLU(),
            nn.Linear(128, in_dim)
        )
        
    def forward(self, x):
        return self.net(x)


In [ ]:
def sample_score_langevin(model, num_samples, in_dim=2, num_steps=500, alpha=0.01):
    model.eval()
    # TODO: Implement Langevin MCMC sampling for score models supporting dynamic in_dim
    # Initialize samples uniformly in [-4, 4] inside R^in_dim.
    # Update: x = x + 0.5 * alpha * s_theta(x) + sqrt(alpha) * N(0, I)
    x = torch.FloatTensor(num_samples, in_dim).uniform_(-4, 4).to(device)

    with torch.no_grad():
        for _ in range(num_steps):
            score = model(x)
            x = x + 0.5 * alpha * score + (alpha ** 0.5) * torch.randn_like(x)

    return get_numpy(x)

def esm_loss(model, x):
    # TODO: Implement ESM loss supporting dynamic in_dim
    x.requires_grad_(True)
    # 1. Forward pass to get score predictions s = model(x).
    # 2. For each dimension d up to in_dim, compute sum of s[:, d].
    # 3. Use torch.autograd.grad to get gradient of the sum with respect to x.
    # 4. Accumulate the diagonal components to compute the Jacobian trace.
    # 5. Return mean loss: (0.5 * ||s||^2 + tr_jac).mean()
    s = model(x)

    in_dim = x.shape[1]
    trace = 0.0
    for d in range(in_dim):
        grad = torch.autograd.grad(s[:, d].sum(), x, create_graph=True)[0]
        trace = trace + grad[:, d]

    loss = 0.5 * (s ** 2).sum(dim=1) + trace
    return loss.mean()

def train_esm(train_data, val_data, dset_type):
    import torch.optim as optim
    from torch.utils.data import DataLoader, TensorDataset
    
    in_dim = train_data.shape[1]
    model = ScoreNet(in_dim=in_dim).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    
    train_dataset = TensorDataset(torch.from_numpy(train_data))
    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
    
    train_losses = []
    val_losses = []
    
    epochs = 30 if dset_type == 1 else 20
    for epoch in range(epochs):
        model.train()
        epoch_losses = []
        for (x_batch,) in train_loader:
            x_batch = x_batch.to(device)
            loss = esm_loss(model, x_batch)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_losses.append(loss.item())
            
        model.eval()
        epoch_val_losses = []
        val_loader = DataLoader(TensorDataset(torch.from_numpy(val_data)), batch_size=128, shuffle=False)
        for (x_val_batch,) in val_loader:
            x_val_batch = x_val_batch.to(device)
            val_loss = esm_loss(model, x_val_batch)
            epoch_val_losses.append(val_loss.item())
            
        train_losses.append(np.mean(epoch_losses))
        val_losses.append(np.mean(epoch_val_losses))
        
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1:02d} | Train ESM Loss: {train_losses[-1]:.4f} | Val ESM Loss: {val_losses[-1]:.4f}")
            
    langevin_samples = sample_score_langevin(model, 1000, in_dim=in_dim, num_steps=500, alpha=0.01)
    return model, train_losses, val_losses, langevin_samples

In [8]:
# Train ESM on Dataset 1 (2D Pinwheel)
esm_model_1 = q2_a_save_results(1, train_data_1, val_data_1, train_esm)

# Train ESM on Dataset 2 (3D GMM)
esm_model_2 = q2_a_save_results(2, train_data_2, val_data_2, train_esm)


Training Score Model with Explicit Score Matching on Dataset 1 (Pinwheel (2D))...


Epoch 05 | Train ESM Loss: -2.2336 | Val ESM Loss: -3.1146


Epoch 10 | Train ESM Loss: -36.8752 | Val ESM Loss: -39.3166


Epoch 15 | Train ESM Loss: -76.2265 | Val ESM Loss: -78.1031


Epoch 20 | Train ESM Loss: -94.4740 | Val ESM Loss: -94.6778


...

Epoch 05 | Train ESM Loss: -4.8018 | Val ESM Loss: -5.1064


Epoch 10 | Train ESM Loss: -5.6092 | Val ESM Loss: -5.7830


Epoch 15 | Train ESM Loss: -5.7904 | Val ESM Loss: -5.8202


Epoch 20 | Train ESM Loss: -5.7456 | Val ESM Loss: -5.8306
Final Train Loss: -5.7456
Final Val Loss: -5.8306


Q2(a) (Dataset 2) results saved successfully!


### Question 2(b): Denoising Score Matching & Noise Level Trade-offs
Explicit Score Matching is computationally expensive due to the Jacobian trace. Denoising Score Matching (DSM) avoids this by perturbing data with noise and matching the score of the noisy distribution:
$$\mathcal{L}_{DSM}(\theta; \sigma) = \frac{1}{2} \mathbb{E}_{x \sim p_{data}, \epsilon \sim \mathcal{N}(0, \sigma^2 I)} \left[ \| s_\theta(x + \epsilon) + \frac{\epsilon}{\sigma^2} \|_2^2 \right]$$

Here, the noise level $\sigma$ is fixed during training and inference. You will analyze how this noise parameter $\sigma$ introduces a trade-off:
- **High noise level**: yields smoother density, making MCMC mixing easier (bridging modes), but provides a worse approximation of the clean data.
- **Low noise level**: matches clean data more closely, but suffers from vanishing score information in low-density space, making MCMC mixing poor.

### Tasks:
1. Implement Denoising Score Matching loss (`dsm_loss`).


In [ ]:
def dsm_loss(model, x, sigma):
    # TODO: Implement DSM loss
    # 1. Sample Gaussian noise with standard deviation = sigma.
    # 2. Compute noisy input x_noisy = x + noise.
    # 3. Predict scores: s = model(x_noisy).
    # 4. Compute target score: -noise / (sigma^2).
    # 5. Return 0.5 * mean of squared differences: 0.5 * ((s - target)^2).sum(dim=1).mean()
    noise = torch.randn_like(x) * sigma
    x_noisy = x + noise
    s = model(x_noisy)
    target = -noise / (sigma ** 2)
    return 0.5 * ((s - target) ** 2).sum(dim=1).mean()

def train_dsm(train_data, val_data, sigma, dset_type):
    import torch.optim as optim
    from torch.utils.data import DataLoader, TensorDataset
    
    in_dim = train_data.shape[1]
    model = ScoreNet(in_dim=in_dim).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    
    train_dataset = TensorDataset(torch.from_numpy(train_data))
    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
    
    train_losses = []
    val_losses = []
    
    epochs = 30 if dset_type == 1 else 20
    for epoch in range(epochs):
        model.train()
        epoch_losses = []
        for (x_batch,) in train_loader:
            x_batch = x_batch.to(device)
            loss = dsm_loss(model, x_batch, sigma)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_losses.append(loss.item())
            
        model.eval()
        epoch_val_losses = []
        with torch.no_grad():
            val_loader = DataLoader(TensorDataset(torch.from_numpy(val_data)), batch_size=128, shuffle=False)
            for (x_val_batch,) in val_loader:
                x_val_batch = x_val_batch.to(device)
                val_loss = dsm_loss(model, x_val_batch, sigma)
                epoch_val_losses.append(val_loss.item())
                
        train_losses.append(np.mean(epoch_losses))
        val_losses.append(np.mean(epoch_val_losses))
        
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1:02d} | Train DSM Loss: {train_losses[-1]:.4f} | Val DSM Loss: {val_losses[-1]:.4f}")
            
    langevin_samples = sample_score_langevin(model, 1000, in_dim=in_dim, num_steps=500, alpha=0.01)
    return model, train_losses, val_losses, langevin_samples

In [10]:
# Train DSM on Dataset 1 (2D Pinwheel) with noise levels: sigma = 0.05, 0.8, 1.5
dsm_pin_005 = q2_b_save_results(1, train_data_1, val_data_1, train_dsm, sigma=0.05)
dsm_pin_08  = q2_b_save_results(1, train_data_1, val_data_1, train_dsm, sigma=0.8)
dsm_pin_15  = q2_b_save_results(1, train_data_1, val_data_1, train_dsm, sigma=1.5)


Training Score Model with Denoising Score Matching (sigma=0.05) on Dataset 1 (Pinwheel (2D))...


Epoch 05 | Train DSM Loss: 395.0382 | Val DSM Loss: 395.9763


Epoch 10 | Train DSM Loss: 399.5214 | Val DSM Loss: 404.9865


Epoch 15 | Train DSM Loss: 406.6323 | Val DSM Loss: 404.7610


Epoch 20 | Train DSM Loss: 408.4182 | Val DSM Loss: 382.9688


...

Epoch 15 | Train DSM Loss: 0.2518 | Val DSM Loss: 0.2661


Epoch 20 | Train DSM Loss: 0.2625 | Val DSM Loss: 0.2504


Epoch 25 | Train DSM Loss: 0.2532 | Val DSM Loss: 0.2622


Epoch 30 | Train DSM Loss: 0.2560 | Val DSM Loss: 0.2670
Final Train Loss: 0.2560
Final Val Loss: 0.2670


Q2(b) sigma=1.5 (Dataset 1) results saved successfully!


In [11]:
# Train DSM on Dataset 2 (3D GMM) with noise levels: sigma = 0.1, 1.45, 2.8
dsm_gmm_01  = q2_b_save_results(2, train_data_2, val_data_2, train_dsm, sigma=0.1)
dsm_gmm_145 = q2_b_save_results(2, train_data_2, val_data_2, train_dsm, sigma=1.45)
dsm_gmm_28  = q2_b_save_results(2, train_data_2, val_data_2, train_dsm, sigma=2.8)


Training Score Model with Denoising Score Matching (sigma=0.1) on Dataset 2 (GMM (3D))...


Epoch 05 | Train DSM Loss: 147.0720 | Val DSM Loss: 142.9123


Epoch 10 | Train DSM Loss: 149.5436 | Val DSM Loss: 146.3995


Epoch 15 | Train DSM Loss: 146.2633 | Val DSM Loss: 145.3929


Epoch 20 | Train DSM Loss: 145.1241 | Val DSM Loss: 142.1623
Final Train Loss: 145.1241
Final Val Loss: 142.1623


...

Epoch 05 | Train DSM Loss: 0.0607 | Val DSM Loss: 0.0628


Epoch 10 | Train DSM Loss: 0.0610 | Val DSM Loss: 0.0585


Epoch 15 | Train DSM Loss: 0.0598 | Val DSM Loss: 0.0596


Epoch 20 | Train DSM Loss: 0.0622 | Val DSM Loss: 0.0627
Final Train Loss: 0.0622
Final Val Loss: 0.0627


Q2(b) sigma=2.8 (Dataset 2) results saved successfully!
